# RSNA Knee Abnormality Detection — Phase 4G gold58 three-plane fusion OOF v1

Internet-off Kaggle notebook. **Do not enable internet. GPU (Tesla T4) is required.**
This notebook trains **one** image-only ResNet-18 OOF model on the **58 fully
labelled training studies** only. It does **not** process report-only studies,
does **not** run the second reader, does **not** touch hidden-test images, and
does **not** submit.

Create a **new** kernel slug `rsna-knee-gold58-3plane-oof-v1`.
Do **not** overwrite frozen kernels (gold-pilot v7, constrained-canary v3,
gold-constrained-v14 v1, gold58-image-oof-v1 v1, gold58-image-oof-v2 v1, SSL canary v1–v4, notebooks 07/09/10/11/12/13/14/15/16/17).

Fail-closed preflight: incomplete PatientID → `GROUPING_PREFLIGHT_FAIL`;
zero readable series under `deterministic-primary-series-v2` → `SERIES_SELECTION_PREFLIGHT_FAIL`;
missing/mismatched ImageNet weights → `IMAGENET_WEIGHTS_PREFLIGHT_FAIL`.
If any gate fails, stop without training.
`HARD_WEAK_LABEL_TRAINING_ALLOWED=false`.
`SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED=false`.
`SECOND_READER_SCORE_BEARING_ALLOWED=false`.
`gold_second_reader_rerun_authorized=false`.


In [ ]:
# Bootstrap: stdlib only until the package is importable.
# Copies/extracts source under /kaggle/working. Never writes under /kaggle/input.
from __future__ import annotations

import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
SRC_DST = KAGGLE_WORKING / "rsna_knee_src"
OUTPUT_DIR = KAGGLE_WORKING / "rsna_gold58_3plane_oof_v1"
MARKERS = ("knee/config.py", "knee/gold58_3plane_oof.py")
WEIGHT_NAME = "resnet18-f37072fd.pth"

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TORCH_HOME"] = str(KAGGLE_WORKING / "torch_cache")


def _looks_like_source(root: Path) -> bool:
    return all((root / marker).is_file() for marker in MARKERS)


def _safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not name or name.endswith("/"):
                continue
            target = (dest / name).resolve()
            try:
                target.relative_to(dest_resolved)
            except ValueError as exc:
                raise RuntimeError(f"refusing zip path that escapes {dest}: {name}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open("wb") as out:
                shutil.copyfileobj(src, out)


def _copy_allowlisted(src_root: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for dirname in ("knee", "scripts", "configs", "notebooks", "vendor", "third_party"):
        src = src_root / dirname
        if src.is_dir():
            shutil.copytree(src, dest / dirname, dirs_exist_ok=True)
    for filename in ("pyproject.toml", "requirements.txt", "README.md", "literature/ontology.yaml"):
        src = src_root / filename
        if src.is_file():
            dest_file = dest / filename
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest_file)


def _zip_has_source(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
    return any(any(name.endswith(marker) for marker in MARKERS) for name in names)


SKIP_WALK_DIRS = {"train_series", "test_series", "__pycache__", ".git", "competitions"}


def _walk_input_for_source(root: Path, max_depth: int = 6):
    zips = []
    dirs = []
    try:
        root_res = root.resolve()
    except OSError:
        return zips, dirs
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        try:
            rel = current.resolve().relative_to(root_res)
            depth = 0 if str(rel) == "." else len(rel.parts)
        except (OSError, ValueError):
            dirnames[:] = []
            continue
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP_WALK_DIRS and d != COMPETITION_SLUG)
        if depth > max_depth:
            dirnames[:] = []
            continue
        if _looks_like_source(current):
            dirs.append(current)
        for name in filenames:
            if name.lower().endswith(".zip"):
                zip_path = current / name
                if _zip_has_source(zip_path):
                    zips.append(zip_path)
    return zips, dirs


def locate_and_extract_source():
    if _looks_like_source(SRC_DST):
        return SRC_DST
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _looks_like_source(candidate):
            if KAGGLE_WORKING.exists() and candidate != SRC_DST:
                _copy_allowlisted(candidate, SRC_DST)
                if _looks_like_source(SRC_DST):
                    return SRC_DST
            return candidate
    if not KAGGLE_INPUT.is_dir():
        return None
    zips = []
    dirs = []
    for child in sorted(KAGGLE_INPUT.iterdir()):
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            continue
        if child.is_file() and child.suffix.lower() == ".zip" and _zip_has_source(child):
            zips.append(child)
        elif child.is_dir():
            found_zips, found_dirs = _walk_input_for_source(child)
            zips.extend(found_zips)
            dirs.extend(found_dirs)
    if zips:
        _safe_extract(zips[0], SRC_DST)
        nested = [p for p in SRC_DST.rglob("knee") if (p.parent / "knee" / "gold58_oof.py").is_file()]
        if _looks_like_source(SRC_DST):
            return SRC_DST
        if nested:
            return nested[0].parent
    if dirs:
        _copy_allowlisted(dirs[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        return dirs[0]
    return None


SOURCE_ROOT = locate_and_extract_source()
print("source_root=" + str(SOURCE_ROOT))
if SOURCE_ROOT is None:
    raise SystemExit("SOURCE_NOT_FOUND: attach mahdimzafari/cursor-knee-source")
sys.path.insert(0, str(SOURCE_ROOT))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("output_dir=" + str(OUTPUT_DIR))


In [ ]:
import json
import os
import time
from pathlib import Path

from knee.gold58_3plane_oof import KERNEL_SLUG, GOLD58_3PLANE_OOF_EXECUTION_ALLOWED, run_gold58_3plane_oof
from knee.kaggle_runtime import KaggleLayout, preflight
from knee.phase4d_readiness import SECOND_READER_SCORE_BEARING_ALLOWED
from knee.report_labels import (
    HARD_WEAK_LABEL_TRAINING_ALLOWED,
    SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED,
)

assert HARD_WEAK_LABEL_TRAINING_ALLOWED is False
assert SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED is False
assert SECOND_READER_SCORE_BEARING_ALLOWED is False
print("kernel_slug=" + KERNEL_SLUG)
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-gold-pilot")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-constrained-canary")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-gold-constrained-v14")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-gold58-image-oof-v1")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-gold58-image-oof-v2")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-unlabeled-image-ssl-canary-v1")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-unlabeled-image-ssl-canary-v2")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-unlabeled-image-ssl-canary-v3")
print("notebook_10_executed=false")
print("notebook_07_executed=false")
print("notebook_09_executed=false")
print("gold_second_reader_rerun_authorized=false")
print("HARD_WEAK_LABEL_TRAINING_ALLOWED=false")
print("SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED=false")
print("SECOND_READER_SCORE_BEARING_ALLOWED=false")

layout = KaggleLayout(output_dirname="rsna_gold58_3plane_oof_v1")
pre = preflight(layout=layout, output_dir=OUTPUT_DIR)
print(pre.message)
print("data_source=", pre.data_source)
if not pre.ok or pre.data_root is None:
    raise SystemExit("DATA_NOT_AVAILABLE: attach rsna-knee-abnormality-detection")

try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    print("gpu_name=" + gpu_name)
    if gpu_name == "cpu":
        raise SystemExit("PREFLIGHT_FAIL: cuda_required")
except Exception as exc:
    raise SystemExit("PREFLIGHT_FAIL: cuda_required:" + str(exc))

cfg_path = str(Path(SOURCE_ROOT) / "configs" / "gold58_image_resnet18_3plane_oof_v1.yaml") if SOURCE_ROOT else None
started = time.time()
result = run_gold58_3plane_oof(
    layout=layout,
    data_root=pre.data_root,
    output_dir=OUTPUT_DIR,
    config_path=cfg_path,
    execute_training=True,
    allow_tiny_encoder_for_tests=False,
    require_cuda=True,
)
elapsed = round(time.time() - started, 3)
peak_gpu = None
try:
    import torch
    if torch.cuda.is_available():
        peak_gpu = int(torch.cuda.max_memory_allocated())
except Exception:
    peak_gpu = None
public = {
    "status": result.get("status"),
    "ok": result.get("ok"),
    "skip_reason": result.get("skip_reason"),
    "training_run": result.get("training_run"),
    "scientific_evaluation": result.get("scientific_evaluation"),
    "n_labelled_studies": result.get("n_labelled_studies"),
    "n_report_only_studies_processed": result.get("n_report_only_studies_processed"),
    "macro_roc_auc": result.get("macro_roc_auc"),
    "bootstrap": result.get("bootstrap"),
    "per_target": result.get("per_target"),
    "preflight_aggregates": result.get("preflight_aggregates"),
    "config_sha256": result.get("config_sha256"),
    "imagenet_weights_sha256": result.get("imagenet_weights_sha256"),
    "checkpoint_sha256": result.get("checkpoint_sha256"),
    "oof_predictions_sha256": result.get("oof_predictions_sha256"),
    "round_trip": result.get("round_trip"),
    "competition_submission_written": result.get("competition_submission_written"),
    "kernel_slug": result.get("kernel_slug"),
    "wall_seconds": elapsed,
    "peak_gpu_memory_bytes": peak_gpu,
    "gpu_name": gpu_name,
}
print("public=" + json.dumps(public, sort_keys=True, default=str)[:12000])
print("GOLD58_3PLANE_OOF_DONE status=" + str(result.get("status")))
print("DO_NOT_SUBMIT")
print("notebook_07_executed=false")
print("notebook_17_executed=false")
print("UNLABELED_IMAGE_SSL_EXECUTION_ALLOWED=false")
print("SSL_ENGINEERING_PATH_PARKED=true")
assert GOLD58_3PLANE_OOF_EXECUTION_ALLOWED is True
if str(result.get("status") or "").endswith("PREFLIGHT_FAIL") or result.get("status") == "PREFLIGHT_FAIL":
    raise SystemExit("PREFLIGHT_FAIL: " + str(result.get("skip_reason")))
